# CTMatch Evaluation Baseline

Establishes baseline metrics on TREC 2021, TREC 2022, and KZ datasets using the existing pipeline.
This is the regression gate — all subsequent changes must match or beat these numbers.

**Filters tested (set in Config cell):**
- `sim` — embedding + category similarity (MiniLM-L6-v2 + bart-large-mnli)
- `svm` — linear SVM on embeddings
- `classifier` — fine-tuned SciBERT

**Metrics:** NDCG@10, MRR, F1, FPR

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q sentence-transformers datasets scikit-learn transformers tqdm evaluate lxml
!pip install -q sympy==1.13.1  # pin for torch compatibility

In [ ]:
# ======== CONFIG ========
# Toggle datasets on/off
USE_TREC_2021 = True
USE_TREC_2022 = True
USE_KZ        = True

# Filter configurations to evaluate — comment out any you don't want
# NOTE: sim filter is for full-corpus retrieval (374k docs → ~100).
# For judged-pool eval it's redundant and slow — use svm+clf only.
FILTER_CONFIGS = {
    'svm+clf':         ['svm', 'classifier'],
    'sim+svm+clf':     ['sim', 'svm', 'classifier'],
}

RUN_DETAILED_EVAL = True   # save per-example JSONL for error analysis

MAX_TOPICS = None   # None = all topics across active datasets

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/ct_data23/evaluation'

# Option 2: Local
# DATA_ROOT = os.environ.get('CTMATCH_DATA_ROOT', '../data')

TREC21_TOPIC_PATH = os.path.join(DATA_ROOT, 'trec_data/trec_21_topics.xml')
TREC21_REL_PATH   = os.path.join(DATA_ROOT, 'trec_data/trec_21_judgments.txt')
TREC22_TOPIC_PATH = os.path.join(DATA_ROOT, 'trec_data/topics2022.xml')
TREC22_REL_PATH   = os.path.join(DATA_ROOT, 'trec_data/qrels2022.txt')
KZ_TOPIC_PATH     = os.path.join(DATA_ROOT, 'kz_data/topics-2014_2015-description.topics')
KZ_REL_PATH       = os.path.join(DATA_ROOT, 'kz_data/qrels-clinical_trials.txt')

# Build active paths from config
trec_topic_paths, rel_paths = [], []
if USE_TREC_2021: trec_topic_paths.append(TREC21_TOPIC_PATH); rel_paths.append(TREC21_REL_PATH)
if USE_TREC_2022: trec_topic_paths.append(TREC22_TOPIC_PATH); rel_paths.append(TREC22_REL_PATH)
if USE_KZ:        rel_paths.append(KZ_REL_PATH)
kz_topic_path = KZ_TOPIC_PATH if USE_KZ else None

print('Active datasets:')
for p in trec_topic_paths + ([kz_topic_path] if kz_topic_path else []) + rel_paths:
    print(f"  {'✓' if os.path.exists(p) else '✗'} {p}")

## Run evaluations

In [ ]:
from ctmatch.evaluation.evaluator import EvaluatorConfig, Evaluator
from ctmatch.matching.pipeline import CTMatch
from ctmatch.config import PipeConfig

# Build CTMatch once — loads embeddings, SVM, SciBERT into memory
ctm = CTMatch(PipeConfig(ir_setup=True))

# Build Evaluator once — setup() creates its own CTMatch which we immediately replace
cfg = EvaluatorConfig(
    rel_paths=rel_paths,
    trec_topic_path=trec_topic_paths or None,
    kz_topic_path=kz_topic_path,
    max_topics=MAX_TOPICS,
)
ev = Evaluator(cfg)
ev.ctm = ctm  # replace with shared instance — no more reloading

all_results = {}
for name, filters in FILTER_CONFIGS.items():
    print(f'\n--- {name} ---')
    ev.ctm.pipe_config = ev.ctm.pipe_config._replace(filters=filters)
    res = ev.evaluate()
    all_results[name] = res
    for k, v in res.items():
        if isinstance(v, float):
            print(f'  {k}: {v:.4f}')

## Detailed evaluation with per-example predictions

Saves every (topic, doc, predicted, actual) to JSONL for error analysis.
Controlled by `RUN_DETAILED_EVAL` in config.

In [ ]:
if RUN_DETAILED_EVAL:
    best_filters = list(FILTER_CONFIGS.values())[-1]  # use last (most complete) config
    cfg_detailed = EvaluatorConfig(
        rel_paths=rel_paths,
        trec_topic_path=trec_topic_paths or None,
        kz_topic_path=kz_topic_path,
        max_topics=MAX_TOPICS,
        filters=best_filters,
    )
    ev_detailed = Evaluator(cfg_detailed)
    detailed_results = ev_detailed.evaluate_detailed(output_path='eval_predictions.jsonl')
    print(f"Saved {detailed_results['total_examples']} predictions ({detailed_results['total_errors']} errors)")
    for k, v in detailed_results.items():
        if isinstance(v, float):
            print(f'  {k}: {v:.4f}')

In [ ]:
import json
with open('eval_predictions.jsonl') as f:
    lines = f.readlines()
print(f'{len(lines)} total predictions')
errors = [json.loads(l) for l in lines if json.loads(l)['is_error']]
print(f'{len(errors)} errors')
print('\nFirst 3 errors:')
for ex in errors[:3]:
    print(f"  topic={ex['topic_id']} doc={ex['doc_id']} pred={ex['predicted_label']} actual={ex['actual_label']}")
    print(f"    topic: {ex['topic_text'][:100]}...")
    print(f"    doc:   {ex['doc_text'][:100]}...")
    print()

In [ ]:
# Download predictions from Colab
# from google.colab import files
# files.download('eval_predictions.jsonl')

## Summary table

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results).T[['ndcg@10', 'mean_mrr', 'mean_f1', 'mean_fpr']]
df.columns = ['NDCG@10 ↑', 'MRR ↑', 'F1 ↑', 'FPR ↓']
df.round(4)

## Previous results reference

From original pipeline (OpenAI text-davinci-003 gen filter, 30 topics):
- `mean_fpr: 7.0`
- `mean_mrr: 0.218`
- `mean_f1: 0.224`

Modernized baseline (TREC 2021 + KZ, sim+svm+clf, 49 topics):
- `ndcg@10: 0.6525`
- `mean_mrr: 0.305`
- `mean_f1: 0.335`